# Lesson 04 — GPU Profiling & Mixed Precision Training

Covers: diagnosing GPU bottlenecks, `torch.profiler`, mixed precision with `torch.cuda.amp`, gradient checkpointing.

## 1. Diagnosing Training Inefficiencies — Framework

In [ ]:
# The four common bottlenecks (check in this order):
#
# 1. DATA LOADING — GPU idle waiting for batches
#    Symptom: GPU utilisation low (<60%), CPU at 100%
#    Fix: increase num_workers, use pin_memory=True, prefetch_factor
#
# 2. GPU MEMORY BOUND — batch size too small or model too large
#    Symptom: small batch, memory usage maxed, utilisation moderate
#    Fix: gradient checkpointing, mixed precision, smaller model
#
# 3. COMPUTE BOUND — genuine arithmetic bottleneck (ideal!)
#    Symptom: GPU util >90%, memory not maxed
#    Fix: larger batch size, torch.compile, cudnn.benchmark
#
# 4. PYTHON/CPU OVERHEAD — too many small ops, Python loops
#    Symptom: GPU util spiky, trace shows many tiny CUDA kernels
#    Fix: vectorise ops, use torch.compile, fuse operations

import torch

# Always set this for CNN workloads (finds fastest conv algorithm)
torch.backends.cudnn.benchmark = True


## 2. DataLoader Profiling

In [ ]:
import time
from torch.utils.data import DataLoader, TensorDataset

X = torch.randn(10000, 3, 64, 64)
y = torch.randint(0, 10, (10000,))
ds = TensorDataset(X, y)

def time_loader(num_workers, pin_memory):
    dl = DataLoader(ds, batch_size=128, num_workers=num_workers,
                    pin_memory=pin_memory, persistent_workers=(num_workers > 0))
    start = time.perf_counter()
    for xb, yb in dl:
        pass  # simulate loading
    elapsed = time.perf_counter() - start
    print(f"  num_workers={num_workers}, pin_memory={pin_memory}: {elapsed:.2f}s")

print("DataLoader timing:")
time_loader(0, False)   # baseline
time_loader(2, True)    # better
time_loader(4, True)    # often optimal (CPU cores / 2)


## 3. Mixed Precision Training with `torch.cuda.amp`

In [ ]:
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

# Mixed precision: forward/loss in fp16, optimizer step in fp32
# Benefit: ~2x speedup on tensor cores, ~50% memory reduction
# Risk: fp16 underflow for small gradients -> GradScaler handles this

def train_with_amp(model, loader, optimizer, criterion, device):
    scaler = GradScaler()   # scales loss to prevent fp16 underflow
    model.train()

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()

        with autocast():            # forward pass in fp16
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()       # scale loss -> scale gradients
        scaler.unscale_(optimizer)          # unscale for clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)              # optimizer step in fp32
        scaler.update()                     # adjust scale for next iter

# Without CUDA, demonstrate the API
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")
print("GradScaler initial scale:", GradScaler().get_scale())


## 4. Gradient Checkpointing (Trading Compute for Memory)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

# Without checkpointing: all activations kept in memory for backward
# With checkpointing:    activations discarded, recomputed during backward
# Trade-off: ~30-40% more compute, ~60-70% less activation memory

class DeepNet(nn.Module):
    def __init__(self, depth=8, d=256):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(depth)])

    def forward_with_checkpoint(self, x):
        for layer in self.layers:
            # Recompute activations during backward instead of storing them
            x = checkpoint(lambda inp, l=layer: torch.relu(l(inp)), x,
                          use_reentrant=False)
        return x

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

net = DeepNet()
x = torch.randn(32, 256)

# Compare memory usage conceptually
print("Forward (no checkpoint): all intermediate activations stored")
out = net(x)
print(f"Output shape: {out.shape}")

print("Forward (with checkpoint): activations recomputed on backward pass")
out_ckpt = net.forward_with_checkpoint(x)
print(f"Output shape: {out_ckpt.shape}")


## 5. `torch.profiler` — Finding the Real Bottleneck

In [ ]:
import torch
import torch.nn as nn
from torch.profiler import profile, record_function, ProfilerActivity

model = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
x = torch.randn(64, 512)

with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    profile_memory=True,
) as prof:
    with record_function("model_inference"):
        out = model(x)

# Show top operators by CPU time
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))


## 6. Diagnosing Vanishing/Exploding Gradients

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def check_gradients(model, loader, criterion, device="cpu"):
    """Run one batch and report gradient norms per layer."""
    model.train()
    xb, yb = next(iter(loader))
    xb, yb = xb.to(device), yb.to(device)
    loss = criterion(model(xb), yb)
    loss.backward()

    grad_norms = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norms[name] = param.grad.norm().item()
    return grad_norms

# Detect vanishing gradients: norms near zero in early layers
# Detect exploding gradients: norms >> 1, loss becomes NaN
# Fix vanishing: residual connections, better init, layer norm, smaller depth
# Fix exploding: gradient clipping, lower LR, gradient norm monitoring

# Example output interpretation:
example_norms = {
    "layer1.weight": 1e-8,   # <-- VANISHING (should be ~1e-3 to 1e-1)
    "layer2.weight": 5e-3,
    "layer3.weight": 2e-2,
    "layer4.weight": 1e-1,
}
print("Gradient norms (vanishing if < 1e-6):")
for k, v in example_norms.items():
    status = "VANISHING" if v < 1e-6 else "OK"
    print(f"  {k}: {v:.2e}  [{status}]")


## Interview Q&A

**Q: GPU utilisation is 30%. What do you check first?**  
Data loading bottleneck — CPU can't feed batches fast enough. Profile with `time_loader`, increase `num_workers`, enable `pin_memory`. Only then look at model/compute.

**Q: Why does mixed precision use a GradScaler?**  
FP16 has range ~6e-5 to 65504. Small gradients (e.g. 1e-8) underflow to zero. GradScaler multiplies the loss by a large factor before `backward()`, shifting gradients into representable fp16 range, then divides optimizer updates back before the weight update.

**Q: When would you NOT use `torch.compile`?**  
Dynamic shapes (variable-length sequences without padding), debugging (compilation hides stack traces), or first-run latency-sensitive code (compilation is expensive). Also PyTorch 2.0+ only.

**Q: `batch_size=32` vs `batch_size=256` — trade-offs?**  
Larger batches: better GPU utilisation, more stable gradient estimates, but may converge to sharper minima (worse generalisation). Smaller batches: noisier gradients act as implicit regularisation. Common practice: scale LR linearly with batch size (linear scaling rule).